# Training the book recommendation model on Kaggle

Before running:

1. **Inputs** (*Add Input → Your Datasets*), both uploaded as **private** datasets:
   - `bookstore-export`: the contents of one `recsys/data/export/<UTC time>/` folder made by `recsys export-dataset`
     (the Goodreads data may not be redistributed, so keep this dataset private);
   - `bookstore-recsys-wheel`: `recsys/dist/recsys-*.whl`, built with `uv build --wheel`.
2. **Settings:** Accelerator *GPU T4 x1* (or P100); Internet *on* (the text model is downloaded from Hugging Face).

Then *Run All*. The model bundle ends up in `/kaggle/working/model.zip` (*Output* tab). Import it into the store with
`uv run recsys import-model model.zip` (step P4).

In [ ]:
import glob
import subprocess
import sys

# Kaggle already has torch, sentence-transformers and scikit-learn; this installs only our package and its
# small dependencies (pandas, pyarrow, psycopg, ...).
wheel = glob.glob("/kaggle/input/bookstore-recsys-wheel/*.whl")[0]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", wheel], check=True)
for package in ("torch", "sentence_transformers", "sklearn"):
    __import__(package)  # fails here, not after an hour, if something is missing

In [ ]:
import torch

print(torch.__version__, "| CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
import pathlib

EXPORT = pathlib.Path(glob.glob("/kaggle/input/bookstore-export/**/meta.json", recursive=True)[0]).parent
OUT = pathlib.Path("/kaggle/working/bundles")
print("Export:", EXPORT)

Training: early stopping on validation NDCG@10 (patience 3), then the test and cold-start evaluation of the model
and of the baselines (popularity, TF-IDF, BPR-MF). The log shows the loss and the validation metrics per epoch.

In [ ]:
!recsys train --data "{EXPORT}" --out "{OUT}" --device cuda --max-epochs 30 --eval-users 3000 --test-users 10000

In [ ]:
import json

import pandas as pd

bundle = sorted(OUT.iterdir())[-1]
metrics = json.loads((bundle / "metrics.json").read_text())
columns = ["ndcg@10", "recall@10", "precision@10", "hr@10", "mrr@10", "coverage@10", "ild@10", "novelty@10"]
rows = {
    (model, part): {c: metrics[model][part].get(c) for c in columns}
    for model in ("hybrid", "popularity", "tfidf", "matrix_factorization")
    if model in metrics
    for part in ("test", "cold_start")
}
display(pd.DataFrame(rows).T.round(4))
display(pd.DataFrame(metrics["training_curve"]).set_index("epoch").round(4))
print("alpha / gamma:", metrics["alpha"])

In [ ]:
import shutil

archive = shutil.make_archive("/kaggle/working/model", "zip", bundle)
print("Download from the Output tab:", archive)